In [1]:
import torch
import torch.nn as nn
import gymnasium as gym
from gymnasium import spaces
import numpy as np
import time
import random
import torch.nn.functional as F
import mujoco
import mujoco.viewer

In [2]:
model_mjcf = """
<mujoco model="pendulum">
  <compiler angle="radian" inertiafromgeom="true"/>
  <option gravity="0 0 -10" timestep="0.05" integrator="RK4"/>

  <worldbody>

    <!-- Fixed pivot + pendulum -->
    <body name="pendulum" pos="0 0 1">
      <!-- Hinge joint (rotation about y-axis) -->
      <joint name="hinge" type="hinge" axis="0 1 0" damping="0.1"/>

      <!-- Rod geometry (visual + collision) -->
      <geom name="rod" type="capsule" fromto="0 0 0 0 0 1"
            size="0.05" rgba="0.8 0.3 0.3 1" mass="1"/>
    </body>
  </worldbody>

  <actuator>
    <!-- Torque motor on the hinge -->
    <motor name="torque" joint="hinge" gear="1" ctrllimited="true" ctrlrange="-2 2"/>
  </actuator>
</mujoco>
"""

In [3]:
class PendulumEnv(gym.Env):
    def __init__(self, render_mode = None):
        self.render_mode = render_mode
        self.model = mujoco.MjModel.from_xml_string(model_mjcf)
        self.data = mujoco.MjData(self.model)
        self.steps = 0
        self.max_torque = 2.0
        self.max_speed = 8.0
        self.max_steps = 200
        self.viewer = None

        high = np.array([1.0, 1.0, self.max_speed], dtype=np.float32)
        self.action_space = spaces.Box(
            low=-self.max_torque, high=self.max_torque, shape=(1,), dtype=np.float32
        )
        self.observation_space = spaces.Box(low=-high, high=high, dtype=np.float32)

    def get_observation(self):
        theta = self.data.qpos[0]
        return np.array(
            [
                np.cos(theta),
                np.sin(theta),
                self.data.qvel[0]
            ],
            dtype=np.float32
        )
    def apply_action(self, action):
        action = np.clip(action, self.action_space.low, self.action_space.high)
        self.data.ctrl[0] = action[0]
        mujoco.mj_step(self.model, self.data)

    def compute_rewards(self):
        theta = (self.data.qpos[0] + np.pi) % (2 * np.pi) - np.pi   # normalize to [-pi, pi]
        reward = -(theta**2 + 0.1 * self.data.qvel[0]**2 + 0.001 * self.data.ctrl[0]**2)
        return reward

    def reset(self):
        self.data.qpos[0] = random.uniform(-np.pi, np.pi)
        self.data.qvel[0] = random.uniform(-1, 1)
        mujoco.mj_forward(self.model, self.data)
        self.steps = 0
        return self.get_observation(), {}
        
    def step(self, action):
        self.apply_action(action)
        next_state = self.get_observation()
        reward = self.compute_rewards()
        self.steps +=1
        truncated = self.steps >= self.max_steps

        if self.render_mode == 'human':
            self.render()
        
        #next_state, reward, terminated, truncated, info =
        return next_state, reward, False, truncated, {}

    def render(self):
        if self.viewer == None:
            self.viewer = mujoco.viewer.launch_passive(self.model, self.data)

        self.viewer.sync()

        if not self.viewer.is_running():
            self.viewer.close()
            self.viewer = None

    def close(self):
        if self.viewer is not None:
            self.viewer.close()
            self.viewer = None

In [4]:
env = PendulumEnv()
env.apply_action(1.0)
print(env.get_observation())

[0.9999935  0.00361072 0.14451262]


In [5]:
from collections import deque

class ReplayBuffer:
    def __init__(self, buffer_size=50000):
        self.buffer = deque(maxlen=buffer_size)

    def push(self, state, action, reward, next_state, done):
        self.buffer.append((state, action, reward, next_state, done))

    def sample(self, batch_size):
        batch = random.sample(self.buffer, batch_size)
        states, actions, rewards, next_states, dones = map(np.stack, zip(*batch))
        return states, actions, rewards, next_states, dones

    def __len__(self):
        return len(self.buffer)

In [6]:
class CriticNetwork(nn.Module):
    def __init__(self, state_dim, action_dim, hidden_dims):
        super().__init__()
        self.critic1= nn.Sequential(
            nn.Linear(state_dim+action_dim, hidden_dims[0]),
            nn.ReLU(),
            nn.Linear(hidden_dims[0], hidden_dims[1]),
            nn.ReLU(),
            nn.Linear(hidden_dims[1], 1),
        )
        self.critic2 = nn.Sequential(
            nn.Linear(state_dim+action_dim, hidden_dims[0]),
            nn.ReLU(),
            nn.Linear(hidden_dims[0], hidden_dims[1]),
            nn.ReLU(),
            nn.Linear(hidden_dims[1], 1),
        )

    def forward(self, state, action):
        inp = torch.cat([state, action], dim=-1)
        Q1 = self.critic1(inp)
        Q2 = self.critic2(inp)
        return Q1, Q2

In [7]:
class ActorNetwork(nn.Module):
    def __init__(self, state_dim, action_dim, hidden_dims, action_limit):
        super().__init__()
        self.action_limit = action_limit
        self.mean_head = nn.Sequential(
                    nn.Linear(state_dim, hidden_dims[0]),
                    nn.ReLU(),
                    nn.Linear(hidden_dims[0], hidden_dims[1]),
                    nn.ReLU(),
                    nn.Linear(hidden_dims[1], action_dim),
                )
        self.log_std_head = nn.Sequential(
                    nn.Linear(state_dim, hidden_dims[0]),
                    nn.ReLU(),
                    nn.Linear(hidden_dims[0], hidden_dims[1]),
                    nn.ReLU(),
                    nn.Linear(hidden_dims[1], action_dim),
                )

    def forward(self, state):
        mean = self.mean_head(state)
        log_std = self.log_std_head(state)

        log_std = torch.clamp(log_std, -20, 2)
        std = torch.exp(log_std)
        dist = torch.distributions.Normal(mean, std)
        sample = dist.rsample()
        action = torch.tanh(sample) 

        log_prob = dist.log_prob(sample)
        log_prob -= torch.log(1-action.pow(2) + 1e-6)
        log_prob = log_prob.sum(dim=-1, keepdim=True)
        action = action * self.action_limit
        
        return action, log_prob

In [ ]:
class SoftActorCritic:
    def __init__(self):
        self.env = PendulumEnv()
        
        self.critic = CriticNetwork(3, 1, (64, 64))
        self.actor = ActorNetwork(3, 1, (64, 64), 2)

        self.target_critic = CriticNetwork(3, 1, (64, 64))
        self.target_critic.load_state_dict(self.critic.state_dict())
        for param in self.target_critic.parameters():
            param.requires_grad = False

        self.replay_buffer = ReplayBuffer()

        self.alpha = 0.1
        self.gamma = 0.98
        self.batch_size = 64

        self.critic_optimizer = torch.optim.Adam(self.critic.parameters(), lr=4e-4)
        self.actor_optimizer = torch.optim.Adam(self.actor.parameters(), lr=4e-4)

    @torch.no_grad()
    def get_target(self, next_states, rewards, dones):
        next_states = torch.as_tensor(next_states, dtype=torch.float32)
        rewards = torch.as_tensor(rewards, dtype=torch.float32).unsqueeze(1)
        dones = torch.as_tensor(dones, dtype=torch.float32).unsqueeze(1)
        next_actions, log_prob = self.actor(next_states)

        target_Q1, target_Q2 = self.target_critic(next_states, next_actions)
        target_Q = torch.min(target_Q1, target_Q2)
        target = rewards + self.gamma * (1-dones) * (
            target_Q - self.alpha * log_prob
        )
        return target

    def update_critic(self, targets, states, actions):
        states = torch.as_tensor(states, dtype=torch.float32)
        actions = torch.as_tensor(actions, dtype=torch.float32)

        Q1, Q2 = self.critic(states, actions)

        critic_loss = F.mse_loss(Q1, targets) + F.mse_loss(Q2, targets)   

        self.critic_optimizer.zero_grad()
        critic_loss.backward()
        self.critic_optimizer.step()

    def update_actor(self, states):
        states = torch.as_tensor(states, dtype=torch.float32)

        new_actions, log_prob = self.actor(states)
        Q1, Q2 = self.critic(states, new_actions)
        Q = torch.min(Q1, Q2)

        actor_loss = (self.alpha * log_prob - Q).mean()

        self.actor_optimizer.zero_grad()
        actor_loss.backward()
        self.actor_optimizer.step()

    def soft_target_update(self, tau=0.005):
        for target_param, critic_param in zip(self.target_critic.parameters(), 
                                            self.critic.parameters()):
            target_param.data.copy_(
                tau * critic_param.data +
                (1-tau) * target_param.data
            )

    def train(self, n_iterations):
        self.actor.train()
        self.critic.train()
        state, _ = self.env.reset()
        state = torch.as_tensor(state, dtype=torch.float32).unsqueeze(0)
        episode_reward = 0
        for step in range(n_iterations):
            if step < 2000:
                action = self.env.action_space.sample()
            else:
                action, _ = self.actor(state)
                action = action.squeeze(0).detach().numpy()
            next_state, reward, terminated, truncated, _ = self.env.step(action)
            episode_reward += reward
            done = terminated or truncated
            self.replay_buffer.push(state.squeeze(0).numpy(), action, reward, next_state, done)

            if len(self.replay_buffer) >= self.batch_size:
                states, actions, rewards, next_states, dones = self.replay_buffer.sample(self.batch_size)
                
                targets = self.get_target(next_states, rewards, dones)
                self.update_critic(targets, states, actions)
                self.update_actor(states)
                self.soft_target_update()

            if done:    
                print(f"Step {step}: {episode_reward:.2f}")
                episode_reward = 0
                state, _ = self.env.reset()
                state = torch.as_tensor(state, dtype=torch.float32).unsqueeze(0)
                
            else:
                state = torch.as_tensor(next_state, dtype=torch.float32).unsqueeze(0)

    @torch.no_grad()
    def eval(self):
        self.actor.eval()
        eval_env = PendulumEnv(render_mode="human")
        state, _ = eval_env.reset()
        state = torch.as_tensor(state, dtype=torch.float32).unsqueeze(0)
        done = False
        while(not done):
            action, _ = self.actor(state)
            action = action.squeeze(0).detach().numpy()
            next_state, reward, terminated, truncated, _ = eval_env.step(action)
            done = terminated or truncated
            state = torch.as_tensor(next_state, dtype=torch.float32).unsqueeze(0)
            time.sleep(0.01)
        eval_env.close()

In [9]:
sac = SoftActorCritic()
# state_dict = torch.load("sac_checkpoint.pt", weights_only=True)
# sac.actor.load_state_dict(state_dict["actor"])
sac.train(100)
sac.eval()

/home/aaron-dsouza/programming/miniconda3/envs/mujoco/lib/python3.12/site-packages/glfw/__init__.py:917: GLFWError: (65548) b'Wayland: The platform does not provide the window position'
  warnings.warn(message, GLFWError)


In [ ]:
for i in range(10):
    sac.eval()